<a href="https://colab.research.google.com/github/WalterCordeiroGuilhermino/Credit-Card-Fraud-Detection-using-PySpark-ML-Pipeline/blob/main/BigDataTrabalho04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# (A)
Nome completo: Walter Cordeiro Guilhermino
Nome no Classroom: Walter Guilhermino

# (B)
URL do Dataset: https://www.kaggle.com/datasets/kartik2112/fraud-detection Tipo de Aprendizado: Classificação
Coluna Target: is_fraud
Features selecionadas: amt, gender, category

In [2]:
# (C)
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("BigDataTrabalho04") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")


path = "fraudTest.csv"


df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv(path)


print("Total de Linhas no Dataset:")
print(df.count())

print("\nPrimeiras 5 linhas:")
df.show(5, truncate=False)

print("\nEstrutura do Schema:")
df.printSchema()



Total de Linhas no Dataset:
555719

Primeiras 5 linhas:
+---+---------------------+----------------+------------------------------------+--------------+-----+------+--------+------+---------------------------+----------+-----+-----+-------+------------------+--------+----------------------+----------+--------------------------------+----------+------------------+-----------+--------+
|_c0|trans_date_trans_time|cc_num          |merchant                            |category      |amt  |first |last    |gender|street                     |city      |state|zip  |lat    |long              |city_pop|job                   |dob       |trans_num                       |unix_time |merch_lat         |merch_long |is_fraud|
+---+---------------------+----------------+------------------------------------+--------------+-----+------+--------+------+---------------------------+----------+-----+-----+-------+------------------+--------+----------------------+----------+--------------------------------+---

In [3]:
# (D)
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler

df_preparado = df.withColumnRenamed("is_fraud", "label")

stringIndexer_gender = StringIndexer(inputCol="gender", outputCol="gender_indexed")
encoder_gender = OneHotEncoder(inputCol="gender_indexed", outputCol="gender_vec")

stringIndexer_category = StringIndexer(inputCol="category", outputCol="category_indexed")
encoder_category = OneHotEncoder(inputCol="category_indexed", outputCol="category_vec")

feature_cols = ["amt", "gender_vec", "category_vec"]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")

In [4]:
# (E)
train_df, valid_df = df_preparado.randomSplit([0.8, 0.2], seed=42)

print("Quantidade de linhas no treino:", train_df.count())
print("Quantidade de linhas na validação:", valid_df.count())

Quantidade de linhas no treino: 444387
Quantidade de linhas na validação: 111332


In [5]:
# (F)

from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

lr = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    maxIter=10
)

pipeline = Pipeline(stages=[
    stringIndexer_gender,
    encoder_gender,
    stringIndexer_category,
    encoder_category,
    assembler,
    lr
])

model = pipeline.fit(train_df)

pred = model.transform(valid_df)

print("Previsões no conjunto de validação:")
pred.select("label", "features", "prediction", "probability").show(10, truncate=False)

# Métrica de Acurácia
evaluator_acc = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)
acc = evaluator_acc.evaluate(pred)
print(f"Acurácia: {acc:.4f}")

# AUC-ROC - Para garantir a eficiência do modelo
evaluator_auc = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)
auc = evaluator_auc.evaluate(pred)
print(f"AUC-ROC: {auc:.4f}")

Previsões no conjunto de validação:
+-----+------------------------------+----------+------------------------------------------+
|label|features                      |prediction|probability                               |
+-----+------------------------------+----------+------------------------------------------+
|0    |(15,[0,1,11],[41.28,1.0,1.0]) |0.0       |[0.9990958544984636,9.041455015363908E-4] |
|0    |(15,[0,1,11],[133.93,1.0,1.0])|0.0       |[0.9989315771349142,0.001068422865085794] |
|0    |(15,[0,5],[4.37,1.0])         |0.0       |[0.9969869110450154,0.003013088954984644] |
|0    |(15,[0,9],[7.93,1.0])         |0.0       |[0.9990654301688239,9.345698311761241E-4] |
|0    |(15,[0,1,10],[5.71,1.0,1.0])  |0.0       |[0.9984493595419905,0.0015506404580094557]|
|0    |(15,[0,1],[6.02,1.0])         |0.0       |[0.9996153373778981,3.846626221019056E-4] |
|0    |(15,[0,1,10],[24.44,1.0,1.0]) |0.0       |[0.998396164240779,0.0016038357592209618] |
|0    |(15,[0,9],[82.32,1.0])     

O pipeline de classificação com base na regressão ligstica foi aplicado com base em um dataset de detecção de fraudes de cartão de crédito, as features utilizadas foram: valor de transação (amt); gênero de titular (gender) e categoria do estabelecimento (category).

É possível obter uma alta acurácia por conta da frequencia das fraudes, que são baixas, portanto o modelo consegue prever a maioria dos casos de fraude. Segundo minhas pesquisas, para garantir uma melhor eficácia seria util utilizar Área sob a Curva ROC ou AU ROC para assegurar a eficiência do modelo criado. Foram selecionadas as variavéis de valor, categoria de consumo e gênero, seguem diretrizes conhecidas onde essas variavéis seguem um padrão em caso de fraude (valores exorbitantes ou de teste, categoria de bens de alta liquidez e valor, e padrões de consumo condicionados a gênero)

